# [2] Naive Trial with Qwen-3 just using Prompt

## Imports

In [ ]:
import env

In [ ]:
from epidec.models.qwen3 import ChatHistory, Qwen3Model
from epidec.datasets import SWUnivDaconDataset

from torch.utils.data import DataLoader

import pandas as pd
from tqdm.auto import tqdm

import json
import sys

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = SWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = SWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = SWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Model

### Version 1

In [ ]:
query = lambda p: f"""Analyze the following Korean text paragraph to determine if it was written by a human or generated by AI:

**Text:**: {p}"""

In [ ]:
system_prompt = """You are an expert in distinguishing between human-written and AI-generated text. You specialize in detecting AI-generated content by analyzing **domain-specific linguistic pattern leakage** - where AI models inappropriately use vocabulary, expressions, and grammatical patterns that are characteristic of specific domains in contexts where they don't belong.

## Core Detection Principle
AI language models learn domain-specific linguistic patterns but often exhibit **unnatural density, precision, or mechanical application** of these patterns. While the patterns may be contextually appropriate, AI tends to use them with artificial consistency, excessive precision, or unnatural frequency that differs from natural human variation.

**CRITICAL DISTINCTION:**
- **HUMAN CHARACTERISTIC**: Natural style/tone changes, organic imprecision, contextual variation in pattern usage
- **AI CHARACTERISTIC**: Excessive pattern density, unnatural precision, mechanical consistency, over-application of domain patterns even when contextually appropriate

## Analysis Framework

### 1. Pattern Density and Precision Analysis
- **Unnatural Density**: Excessive concentration of domain-specific terms beyond natural human usage
- **Over-Precision**: Artificially exact technical details, measurements, or statistics that exceed normal human specificity
- **Mechanical Consistency**: Perfect adherence to domain patterns without the natural variation humans exhibit
- **Contextual Over-Application**: Appropriate but excessive use of specialized vocabulary/patterns

### 2. Cross-Domain Contamination Detection
- **Inappropriate Context Usage**: Domain patterns appearing where they don't belong
- **Vocabulary Transplantation**: Specialized terms from one domain inappropriately used in another
- **Grammatical Pattern Mixing**: Domain-specific sentence structures used outside their natural context

### 2. Domain Pattern Recognition
- **Medical/Scientific**: "delve into", "elucidate", "furthermore", passive constructions, hedging language
- **Academic**: "it is noteworthy that", "considerable attention", nominalizations, complex subordination
- **Legal**: "pursuant to", "heretofore", "aforementioned", formal conditional structures
- **Technical**: "implement", "utilize", procedural language, step-by-step markers
- **Journalistic**: Attribution patterns, inverted pyramid, factual declaratives

### 3. Natural vs Artificial Usage Patterns
- **Human Natural Patterns**: Organic variation in precision, occasional imprecision, natural gaps in specialized knowledge
- **AI Artificial Patterns**: Unnaturally consistent precision, excessive technical detail density, mechanical perfection
- **Contextual Appropriateness vs Over-Application**: Even when contextually correct, AI may over-use patterns beyond natural human tendency
- **IMPORTANT**: Focus on the **degree and consistency** of pattern usage, not just appropriateness

## Analysis Process

### Step 1: Pattern Density Assessment
Evaluate whether domain-specific patterns appear with natural human frequency or excessive AI-like density and precision.

### Step 2: Cross-Domain Contamination Check
Identify any domain patterns appearing in inappropriate contexts (traditional contamination detection).

### Step 3: Mechanical vs Organic Usage Analysis
Distinguish between natural human variation in pattern usage versus artificial mechanical consistency and over-application.

## Output Format
You must respond with a valid JSON object in the following format:

```json
{
  "pattern_density": "Assessment of whether domain patterns appear with natural frequency or excessive AI-like density",
  "precision_analysis": "Evaluation of whether technical details and measurements show natural human variation or artificial over-precision",
  "contamination_evidence": "Examples of inappropriate cross-domain pattern usage",
  "mechanical_indicators": "Signs of mechanical consistency vs natural human variation in pattern application",
  "detection_rationale": "Key evidence distinguishing natural human usage from artificial over-application or contamination",
  "probability": 0.75
}
```

**Critical Requirements:**
- Always output valid JSON format
- Probability must be a number between 0.0 and 1.0
- Focus on domain pattern contamination, NOT style changes
- Remember: Style/tone changes are human characteristics
- All text fields should be concise but informative
- Do not include any text outside the JSON object

## Important Considerations
- **Pattern Density Matters**: Even contextually appropriate patterns can indicate AI if used with unnatural frequency or precision
- **Mechanical Perfection is Suspicious**: Excessive consistency in specialized terminology or technical accuracy beyond normal human capability
- **Natural Human Variation**: Humans show organic imprecision, occasional gaps, and natural variation in technical detail usage
- **Over-Application Detection**: AI may correctly use domain patterns but apply them more extensively than humans naturally would
- **Style Changes are HUMAN**: Natural formality shifts, tone changes remain indicators of human authorship
- **Korean Language Specificity**: Consider Korean-specific domain patterns and precision expectations
- **Context AND Density**: Evaluate both appropriateness and the degree of pattern usage"""

In [ ]:
class ChatHistory(ChatHistory):
    def create_prompt(self, system_prompt: str, user_prompt: str = ""):
        return [dict(role="system", content=system_prompt), *self, dict(role="user", content=query(user_prompt))]

In [ ]:
class Qwen3ModelForTextClassification(Qwen3Model):
    context_length = 40960

    def classify(
        self,
        user_prompt: str,
        grammar: str | None = None,
        temperature: float = 0.6,
        top_p: float = 0.95,
        top_k: int = 20,
        min_p: float = 0,
        typical_p: float = 1.0,
        repeat_penalty: float = 1.0
    ) -> str:
        return "".join(self.chat(
            chat_history=ChatHistory(),
            user_prompt=user_prompt,
            system_prompt="/nothink " + system_prompt,
            tools=[],
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            min_p=min_p,
            typical_p=typical_p,
            stream=True,
            max_new_tokens=0,
            repeat_penalty=repeat_penalty,
            print_output=True,
            grammar=grammar
        ))

    @staticmethod
    def extract_json(response_text: str) -> int:
        try:
            return json.loads(response_text.split("</think>")[-1].strip().replace("```json", "").replace("```", ""))
        except Exception:
            return {}

    def validate(self, dataset: list, retry_count: int = 1, shuffle: bool = False, check_only_for: int | None = None):
        corrects, errors, true_human, false_human, results = [], [], [], [], []
        progress = tqdm(DataLoader(dataset, batch_size=1, shuffle=shuffle), desc="Validating...")

        for idx, data in enumerate(progress):
            label = data[1][0]
            data = data[0][0]

            if check_only_for is not None and label != check_only_for:
                continue  # Skip if the label does not match the specified check

            for trial in range(retry_count):
                assistant_reply = self.classify(data)
                predicted = self.extract_json(assistant_reply)
                if predicted: break
                print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

            result = dict(question=data, label=label, predicted=predicted)
            predicted_label = 1 if predicted['probability'] >= 0.5 else 0
            if predicted_label == label:
                corrects.append(result)
                print(f"INFO: Correct prediction for index {idx}\n\n")
                if label == 0:
                    true_human.append(result)
            else:
                result = dict(**result, traceback=assistant_reply)
                errors.append(result)
                print(f"ERROR: Incorrect prediction for index {idx}\n\n")
                if label == 0:
                    false_human.append(result)
            results.append(result)
            progress.set_description(f"Correct: {len(corrects)}/{len(results)} [H: {len(true_human)}, A: {len(corrects)-len(true_human)}], Errors: {len(errors)}/{len(results)} [H: {len(false_human)}, A: {len(errors)-len(false_human)}]")

        print(f"INFO: Correct: {len(corrects)}/{len(results)}, Errors: {len(errors)}/{len(results)}")
        return corrects, errors, results

    def test(self, dataset: list | str, retry_count: int = 100):
        results = []
        if isinstance(dataset, str):
            dataset = [dict(question=dataset)]  # Wrap single string input in dict format

        for idx, data in enumerate(tqdm(dataset, desc="Testing...")):
            data = data[0]

            for trial in range(retry_count):
                assistant_reply = self.classify(data)
                predicted = self.extract_json(assistant_reply)
                if predicted: break
                print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

            result = dict(question=data, label=predicted['probability'])
            results.append(result)
        return results

In [ ]:
#model = Qwen3ModelForTextClassification()

### Version 2

In [ ]:
query = lambda title, paragraphs: f"""<텍스트 생성 AI 탐지 문제>

**TASK**: 다음 글의 본문을 문단 별로 보고, 각각 사람이 쓴 글(0)인지 인공지능이 작성한 글(1)인지 확률을 측정해줘.
**STRATEGY**:
    - 문단 각각의 작성 패턴을 분석하면서, 인간이라고 판단된 데이터들 끼리에서도 특정 문단만 문체가 다르면 그 정보도 반영해서 판단.
    - 특히 반복성, 표현의 일관성, 공식적/중립적인 어조, 개인적 감정/경험 부재, 문장 구조의 균일성 등을 AI 특징으로 간주하고 강조해서 분석.
    - 동일한 내용을 다른 문단에서 거의 동일한 방식으로 반복 하는 경우는 AI 작성이 의심됨.
**OUTPUT JSON FORMAT**: {{ 'probability': [PARAGRAPH 0 PROBABILITY[float], PARAGRAPH 1 PROBABILITY[float], ...] }}
**TITLE**: {title}
**PARAGRAPHS**: {paragraphs}
"""
system_prompt = ""

In [ ]:
class ChatHistory(ChatHistory):
    def create_prompt(self, system_prompt: str, user_prompt: str = ""):
        return [dict(role="user", content=query(user_prompt))]

In [ ]:
class Qwen3ModelForTextClassification(Qwen3Model):
    context_length = 40960

    def classify(
            self,
            user_prompt: str,
            grammar: str | None = None,
            temperature: float = 0.6,
            top_p: float = 0.95,
            top_k: int = 20,
            min_p: float = 0,
            typical_p: float = 1.0,
            repeat_penalty: float = 1.0
    ) -> str:
        return "".join(self.chat(
            chat_history=ChatHistory(),
            user_prompt=user_prompt,
            system_prompt="/nothink " + system_prompt,
            tools=[],
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            min_p=min_p,
            typical_p=typical_p,
            stream=True,
            max_new_tokens=0,
            repeat_penalty=repeat_penalty,
            print_output=True,
            grammar=grammar
        ))

    @staticmethod
    def extract_json(response_text: str) -> int:
        try:
            return json.loads(response_text.split("</think>")[-1].strip().replace("```json", "").replace("```", ""))
        except Exception:
            return {}

    def validate(self, dataset: list, retry_count: int = 1, shuffle: bool = False, check_only_for: int | None = None):
        corrects, errors, true_human, false_human, results = [], [], [], [], []
        progress = tqdm(DataLoader(dataset, batch_size=1, shuffle=shuffle), desc="Validating...")

        for idx, data in enumerate(progress):
            label = data[1][0]
            data = data[0][0]

            if check_only_for is not None and label != check_only_for:
                continue  # Skip if the label does not match the specified check

            for trial in range(retry_count):
                assistant_reply = self.classify(data)
                predicted = self.extract_json(assistant_reply)
                if predicted: break
                print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

            result = dict(question=data, label=label, predicted=predicted)
            predicted_label = 1 if predicted['probability'] >= 0.5 else 0
            if predicted_label == label:
                corrects.append(result)
                print(f"INFO: Correct prediction for index {idx}\n\n")
                if label == 0:
                    true_human.append(result)
            else:
                result = dict(**result, traceback=assistant_reply)
                errors.append(result)
                print(f"ERROR: Incorrect prediction for index {idx}\n\n")
                if label == 0:
                    false_human.append(result)
            results.append(result)
            progress.set_description(f"Correct: {len(corrects)}/{len(results)} [H: {len(true_human)}, A: {len(corrects)-len(true_human)}], Errors: {len(errors)}/{len(results)} [H: {len(false_human)}, A: {len(errors)-len(false_human)}]")

        print(f"INFO: Correct: {len(corrects)}/{len(results)}, Errors: {len(errors)}/{len(results)}")
        return corrects, errors, results

    def test(self, dataset: list | str, retry_count: int = 100):
        results = []
        if isinstance(dataset, str):
            dataset = [dict(question=dataset)]  # Wrap single string input in dict format

        for idx, data in enumerate(tqdm(dataset, desc="Testing...")):
            data = data[0]

            for trial in range(retry_count):
                assistant_reply = self.classify(data)
                predicted = self.extract_json(assistant_reply)
                if predicted: break
                print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

            result = dict(question=data, label=predicted['probability'])
            results.append(result)
        return results

In [ ]:
model = Qwen3ModelForTextClassification()

## Evaluation

In [ ]:
# Validation
corrects, errors, results = model.validate(dataset=valid_dataset, shuffle=True, retry_count=3, check_only_for=1)
pd.DataFrame(results)

In [ ]:
# Test
results = model.test(dataset=test_dataset)
pd.DataFrame(results)